# ArcNeuron trên Google Colab

Notebook này chỉ chạy các file thật trong repo. `train.py` tự đọc lượng data thật và tính context, batch, số step hợp lý; notebook không hardcode 3000 step nữa.

Bật GPU trong **Runtime → Change runtime type → GPU** trước khi chạy.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = "https://github.com/ArcatureLabs/ArcNeuron.git"
ROOT = Path("/content/ArcNeuron")

if not (ROOT / "arcneuron.py").is_file():
    if ROOT.exists():
        subprocess.run(["rm", "-rf", str(ROOT)], check=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)

os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)

def run_live(command):
    """Run a child Python process and stream every output line into this Colab cell."""
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, command)

print("working directory:", Path.cwd())


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU chưa được bật. Chọn Runtime > Change runtime type > GPU rồi chạy lại.")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())
props = torch.cuda.get_device_properties(0)
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB")


## Cấu hình model

Chỉ giữ các quyết định thuộc kiến trúc ở đây. Budget train thuộc về `train.py` và được tính từ `train.txt`.


In [ ]:
VOCAB_SIZE = 1024
DIM = 384
HEADS = 6
KV_HEADS = 2
FFN_DIM = 1024
PRELUDE_LAYERS = 1
CORE_LAYERS = 2
CODA_LAYERS = 1
MAX_DEPTH = 4

BASE_CKPT = "arcneuron.pt"
TUNED_CKPT = "arcneuron-tuned.pt"


## Train base model

Trainer sẽ in params, token count, context, batch, số step, corpus-equivalent exposure, loss, validation loss và ETA theo thời gian thực.


In [ ]:
train_command = [
    sys.executable, "-u", "train.py",
    "--data", "train.txt",
    "--out", BASE_CKPT,
    "--steps", "auto",
    "--batch-size", "auto",
    "--context", "auto",
    "--vocab-size", str(VOCAB_SIZE),
    "--dim", str(DIM),
    "--heads", str(HEADS),
    "--kv-heads", str(KV_HEADS),
    "--ffn-dim", str(FFN_DIM),
    "--prelude-layers", str(PRELUDE_LAYERS),
    "--core-layers", str(CORE_LAYERS),
    "--coda-layers", str(CODA_LAYERS),
    "--max-depth", str(MAX_DEPTH),
]

run_live(train_command)


## Tuning

`tune.py` cũng tự tính số step từ kích thước `tune.txt`; không còn magic 600/1000 step.


In [ ]:
tune_command = [
    sys.executable, "-u", "tune.py",
    "--checkpoint", BASE_CKPT,
    "--data", "tune.txt",
    "--replay-data", "train.txt",
    "--out", TUNED_CKPT,
    "--steps", "auto",
    "--batch-size", "auto",
    "--context", "auto",
    "--max-depth", str(MAX_DEPTH),
]

run_live(tune_command)


## Generate


In [ ]:
from generate import load_model, generate

device = torch.device("cuda")
checkpoint = TUNED_CKPT if Path(TUNED_CKPT).is_file() else BASE_CKPT
model, tokenizer = load_model(checkpoint, device)

PROMPT = "Một con mèo bị mất một chân có còn là động vật có vú không? Giải thích."
DEPTH = 4

answer = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    depth=DEPTH,
    max_new_tokens=128,
    temperature=0.70,
    top_k=40,
    top_p=0.90,
    repetition_penalty=1.08,
    repeat_window=96,
    include_prompt=False,
    device=device,
)

print("Câu hỏi:", PROMPT)
print("Trả lời:", answer)


## So sánh recurrent depth


In [ ]:
for depth in [1, 2, 4, 8]:
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    answer = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=PROMPT,
        depth=depth,
        max_new_tokens=128,
        temperature=0.0,
        top_k=0,
        top_p=1.0,
        repetition_penalty=1.06,
        repeat_window=96,
        include_prompt=False,
        device=device,
    )
    print(f"\n{'=' * 24} depth={depth} {'=' * 24}\n")
    print(answer)


## Tải checkpoint


In [ ]:
from google.colab import files
path = TUNED_CKPT if Path(TUNED_CKPT).is_file() else BASE_CKPT
files.download(path)
